In [3]:
# Make the project root (the parent of 'notebooks/') importable
import sys, pathlib
cwd = pathlib.Path.cwd()
if cwd.name == "backtest":
    proj_root = cwd.parent
else:
    proj_root = cwd  # if you launched from project root
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

In [4]:
# ----------------------------
# Drop-in UPDATE: Regime portfolio backtest (fixes lookahead, missing-data, regime metrics)
# ----------------------------
from __future__ import annotations
from regime_detector import RegimeDetector
from dataclasses import dataclass
from typing import Dict, List, Optional

import itertools
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt


In [2]:
ANN_FACTOR = 252

In [3]:



# ----------------------------
# Load equity curves
# ----------------------------
def load_equity_csv(path: str, name: str, date_col: str = "Date", equity_col: str = "Equity") -> pd.DataFrame:
    """
    CSV columns expected (default): Date, Equity

    Returns Date index with:
      - Equity_<name>
      - Return_<name>  (pct_change of Equity)
    """
    df = pd.read_csv(path, parse_dates=[date_col]).sort_values(date_col)
    df = df[[date_col, equity_col]].rename(columns={date_col: "Date", equity_col: "Equity"})
    df["Equity"] = pd.to_numeric(df["Equity"], errors="coerce")
    df = df.dropna(subset=["Date", "Equity"]).drop_duplicates("Date")
    df = df.set_index("Date").sort_index()
    df["Return"] = df["Equity"].pct_change()
    return df.rename(columns={"Equity": f"Equity_{name}", "Return": f"Return_{name}"})


# ----------------------------
# Metrics (return-based, robust)
# ----------------------------
def compute_perf_metrics_from_returns(ret: pd.Series, ann_factor: int = ANN_FACTOR) -> dict:
    """
    ret: daily returns series (not equity). Index can be irregular; ann_factor still used for scaling.
    """
    ret = pd.Series(ret).dropna()
    n = int(len(ret))
    if n < 30:
        return {"Days": n, "TotalReturn": np.nan, "CAGR": np.nan, "Sharpe": np.nan, "MaxDD": np.nan}

    # Equity curve implied by returns for DD and total return
    eq = (1.0 + ret).cumprod()
    total_return = float(eq.iloc[-1] - 1.0)

    # CAGR from number of periods
    years = n / float(ann_factor)
    cagr = float(eq.iloc[-1] ** (1.0 / years) - 1.0) if years > 0 else np.nan

    r = ret.to_numpy(dtype=float)
    r_std = float(np.std(r, ddof=1))
    sharpe = float((np.mean(r) / r_std) * np.sqrt(ann_factor)) if r_std > 0 else np.nan

    roll_max = eq.cummax()
    max_dd = float((eq / roll_max - 1.0).min())

    return {"Days": n, "TotalReturn": total_return, "CAGR": cagr, "Sharpe": sharpe, "MaxDD": max_dd}


def run_portfolio_regime_test(
    equity_curves: List[dict],
    weights_by_regime: Dict[str, Dict[str, float]],
    detector,  # RegimeDetectorEMA (or compatible: has build_regimes and shift_regime_by_one_day attr)
    initial_equity: float = 1_000_000.0,
    normalize_weights: bool = False,
    # Keep these for parity, but the detector owns most regime params
    shift_regime_by_one_day: bool = True,
    missing_data_mode: str = "drop",   # "drop" | "zero_weight"
) -> dict:
    """
    Same as run_portfolio_regime_test, but regimes come from `detector.build_regimes()`.
    Expects detector.build_regimes() returns a DF with index=Date and column "RegimeLabel".
    """

    if missing_data_mode not in ("drop", "zero_weight"):
        raise ValueError("missing_data_mode must be one of: 'drop', 'zero_weight'")

    strategy_names = [c["name"] for c in equity_curves]

    # Validate weights dict includes all strategies for each regime (0.0 allowed)
    missing = []
    for regime, w in weights_by_regime.items():
        for s in strategy_names:
            if s not in w:
                missing.append((regime, s))
    if missing:
        ex = "\n".join([f"  - regime='{r}' missing strategy='{s}'" for r, s in missing[:10]])
        raise ValueError(
            "weights_by_regime is missing some (regime, strategy) entries. Examples:\n"
            f"{ex}\n\nTip: provide every strategy in every regime (set 0.0 to disable)."
        )

    # Load and merge equity curves
    frames = []
    for c in equity_curves:
        frames.append(
            load_equity_csv(
                path=c["csv"],
                name=c["name"],
                date_col=c.get("date_col", "Date"),
                equity_col=c.get("equity_col", "Equity"),
            )
        )
    df = pd.concat(frames, axis=1).sort_index()
    if df.empty:
        raise RuntimeError("No equity curve data loaded.")

    # Build regimes using detector over the same date window
    start_date, end_date = df.index.min(), df.index.max()
    regimes = detector.build_regimes(start_date=start_date, end_date=end_date)

    # Expect detector output column name
    if "RegimeLabel" not in regimes.columns:
        raise RuntimeError("detector.build_regimes() must return a DataFrame containing 'RegimeLabel' column.")

    # Use consistent naming with the old backtest ("Regime")
    regimes = regimes[["RegimeLabel"]].rename(columns={"RegimeLabel": "Regime"}).copy()

    # IMPORTANT: drop warmup/invalid rows (NaN regimes)
    regimes = regimes.dropna(subset=["Regime"])
    if regimes.empty:
        raise RuntimeError("Detector produced no valid regimes (warmup too short? increase history/start date).")

    # Join regimes onto curve dates (inner => only dates where regimes exist)
    out = df.join(regimes, how="inner")
    if out.empty:
        raise RuntimeError("After joining equity curves with regimes, no overlapping dates were found.")

    # Lookahead fix (keep same semantics as before)
    if shift_regime_by_one_day:
        out["Regime"] = out["Regime"].shift(1)
        out = out.dropna(subset=["Regime"])
        if out.empty:
            raise RuntimeError("After shifting Regime by 1 day, no data remains (check overlap / warmup).")

    # Strategy returns
    ret_cols = [f"Return_{s}" for s in strategy_names]

    # Handle missing sleeve returns honestly
    if missing_data_mode == "drop":
        out = out.dropna(subset=ret_cols, how="any")
        if out.empty:
            raise RuntimeError("After dropping missing sleeve returns, no rows remain. Try missing_data_mode='zero_weight'.")
    else:
        # Keep NaNs for now; we will mask weights for missing sleeves per-row
        pass

    # Build weights matrix per day
    W = np.zeros((len(out), len(strategy_names)), dtype=float)
    reg_arr = out["Regime"].astype(str).to_numpy()

    # Precompute base weights per regime (vector per regime)
    base_w = {}
    for regime_name in weights_by_regime:
        base_w[regime_name] = np.array([float(weights_by_regime[regime_name][s]) for s in strategy_names], dtype=float)

    # Strategy return matrix (may contain NaNs if zero_weight mode)
    R = out[ret_cols].to_numpy(dtype=float)

    for i, regime_name in enumerate(reg_arr):
        if regime_name not in base_w:
            raise KeyError(
                f"Regime '{regime_name}' not found in weights_by_regime keys: {list(base_w.keys())}"
            )

        w = base_w[regime_name].copy()

        if missing_data_mode == "zero_weight":
            nan_mask = ~np.isfinite(R[i, :])
            if np.any(nan_mask):
                w[nan_mask] = 0.0

        if normalize_weights:
            denom = float(np.sum(np.abs(w)))
            if denom > 0:
                w = w / denom

        W[i, :] = w

    # Add weight columns
    for j, s in enumerate(strategy_names):
        out[f"W_{s}"] = W[:, j]

    # Portfolio return
    R_safe = np.where(np.isfinite(R), R, 0.0)
    out["PortfolioReturn"] = np.sum(W * R_safe, axis=1)

    # Portfolio equity
    out["PortfolioEquity"] = float(initial_equity) * (1.0 + out["PortfolioReturn"]).cumprod()

    # Metrics overall
    metrics_overall = compute_perf_metrics_from_returns(out["PortfolioReturn"])

    # Metrics by regime
    metrics_by_regime = {}
    for regime_name in ["Stable Risk-On", "Fragile", "Vol Shock", "Crisis"]:
        mask = out["Regime"].astype(str) == regime_name
        if mask.any():
            metrics_by_regime[regime_name] = compute_perf_metrics_from_returns(out.loc[mask, "PortfolioReturn"])
    metrics_by_regime_df = pd.DataFrame(metrics_by_regime).T if metrics_by_regime else pd.DataFrame()

    weights_table = pd.DataFrame(weights_by_regime).T[strategy_names]

    return {
        "portfolio_df": out.reset_index().rename(columns={"index": "Date"}),
        "metrics_overall": metrics_overall,
        "metrics_by_regime": metrics_by_regime_df,
        "weights_table": weights_table,
        "regimes_params": {
            "detector_class": detector.__class__.__name__,
            # Best-effort: capture common detector attrs if present
            "vix_high_pct": getattr(detector, "vix_high_pct", None),
            "spread_wide_pct": getattr(detector, "spread_wide_pct", None),
            "lookback": getattr(detector, "lookback", None),
            "credit_mode": getattr(detector, "credit_mode", None),
            "ema_span": getattr(detector, "ema_span", None),
            "shift_regime_by_one_day": shift_regime_by_one_day,
            "missing_data_mode": missing_data_mode,
        },
    }

In [4]:
# 1) Provide your equity curves (CSV paths)
equity_curves = [
    {"name":"trend", "csv":"equity-curves/trend-equity-vt.csv"},
    {"name":"triple_coint", "csv":"equity-curves/pairs_portfolio_equity_curve.csv"},
    {"name":"triple_vol", "csv":"equity-curves/volatility-triple-harvester.csv"},
]

In [5]:

def generate_weight_grid(
    strategies: list[str],
    step: float = 0.10,
    long_only: bool = True,
    sum_to_one: bool = True,
    max_leverage: float = 1.0,   # only used if sum_to_one=False
):
    """
    Generates weight dictionaries for given strategies.

    If sum_to_one=True:
      - produces weights that sum to 1 (within floating tolerance)
    Else:
      - produces weights with sum(abs(w)) <= max_leverage (allows cash if < max_leverage)

    long_only=True => weights in [0,1]
    long_only=False => weights in [-1,1] on a step grid (be careful: explodes combos)
    """
    # Build candidate values per weight
    if long_only:
        vals = np.round(np.arange(0.0, 1.0 + 1e-12, step), 10)
    else:
        vals = np.round(np.arange(-1.0, 1.0 + 1e-12, step), 10)

    candidates = []
    for combo in itertools.product(vals, repeat=len(strategies)):
        w = np.array(combo, dtype=float)

        if sum_to_one:
            if long_only:
                if abs(w.sum() - 1.0) > 1e-9:
                    continue
            else:
                # sum to one in signed sense is unusual; keep if you really want it
                if abs(w.sum() - 1.0) > 1e-9:
                    continue
        else:
            if np.sum(np.abs(w)) > max_leverage + 1e-9:
                continue

        # skip all-zero (possible if sum_to_one=False)
        if np.allclose(w, 0.0):
            continue

        candidates.append(dict(zip(strategies, w.tolist())))

    return candidates

In [6]:
def score_metrics(m: dict, dd_penalty: float = 0.0) -> float:
    """
    Higher is better. Example:
      score = Sharpe - dd_penalty * abs(MaxDD)

    dd_penalty=0.0 => pure Sharpe
    """
    sharpe = m.get("Sharpe", np.nan)
    maxdd = m.get("MaxDD", np.nan)
    if not np.isfinite(sharpe):
        return -np.inf
    if dd_penalty > 0 and np.isfinite(maxdd):
        return float(sharpe - dd_penalty * abs(maxdd))
    return float(sharpe)

In [7]:


def gridsearch_full_regime_weights(
    equity_curves: list[dict],
    detector,
    active_strats_by_regime: dict,
    step: float = 0.10,
    long_only: bool = True,
    sum_to_one_within_regime: bool = True,
    dd_penalty: float = 0.0,
    top_k: int = 20,
    normalize_weights: bool = False,
    shift_regime_by_one_day: bool = True,
    missing_data_mode: str = "drop",
    initial_equity: float = 10_000.0,
    fail_fast: bool = False,
):
    """
    Pure optimizer. No weights_by_regime input required.

    Returns:
        best_weights_by_regime
        results_df (ranked candidates)
    """

    all_strats = [c["name"] for c in equity_curves]
    regimes = list(active_strats_by_regime.keys())

    # Generate weight candidates per regime
    regime_candidates = {}

    for regime in regimes:
        active = active_strats_by_regime[regime]
        candidates = generate_weight_grid(
            strategies=active,
            step=step,
            long_only=long_only,
            sum_to_one=sum_to_one_within_regime,
        )
        regime_candidates[regime] = candidates

    rows = []
    best_score = -np.inf
    best_weights = None


    # Cartesian product across regimes
    for combo in itertools.product(*regime_candidates.values()):

        weights_by_regime = {}

        for regime_name, active_weights in zip(regimes, combo):

            # start all zero
            w_full = {s: 0.0 for s in all_strats}

            # assign active
            for s, v in active_weights.items():
                w_full[s] = float(v)

            weights_by_regime[regime_name] = w_full

        # Run backtest
        res = run_portfolio_regime_test(
            equity_curves=equity_curves,
            weights_by_regime=weights_by_regime,
            detector=detector,
            initial_equity=initial_equity,
            normalize_weights=normalize_weights,
            shift_regime_by_one_day=shift_regime_by_one_day,
            missing_data_mode=missing_data_mode,
        )
        
        m = res["metrics_overall"]
        sc = score_metrics(m, dd_penalty=dd_penalty)

        rows.append({
            "score": sc,
            "Sharpe": m.get("Sharpe", np.nan),
            "CAGR": m.get("CAGR", np.nan),
            "MaxDD": m.get("MaxDD", np.nan),
            "weights": weights_by_regime,
        })

        if sc > best_score:
            best_score = sc
            best_weights = weights_by_regime

    results_df = pd.DataFrame(rows).sort_values("score", ascending=False).head(top_k)

    return best_weights, results_df

In [8]:
active_strats_by_regime = {
    "Stable Risk-On": ["trend"],
    "Fragile":        [
                        "trend", 
                        "triple_coint", 
                       "triple_vol"
                      ],
    "Vol Shock":      [
                        "triple_coint", 
                        ],
    "Crisis":         [
                    "triple_coint", 
            ],
}

detector = RegimeDetector(
    ema_span=45,
    lookback=252,
    vix_high_pct=0.70,
    spread_wide_pct=0.70,
    credit_mode="ratio",
    shift_regime_by_one_day=True,
)
    

best_weights, results_df = gridsearch_full_regime_weights(
    equity_curves=equity_curves,
    detector=detector,
    active_strats_by_regime=active_strats_by_regime,
    step=0.10,              # start coarse (0.10), then refine (0.05)
    dd_penalty=0.0,         # try 0.5 or 1.0 if you want to penalize drawdowns
    sum_to_one_within_regime=True,
    shift_regime_by_one_day=True,
    missing_data_mode="drop",
)



In [9]:
best_weights

{'Stable Risk-On': {'trend': 1.0,
  'triple_coint': 0.0,
  'futures_trend': 0.0,
  'triple_vol': 0.0},
 'Fragile': {'trend': 0.3,
  'triple_coint': 0.5,
  'futures_trend': 0.0,
  'triple_vol': 0.2},
 'Vol Shock': {'trend': 0.0,
  'triple_coint': 1.0,
  'futures_trend': 0.0,
  'triple_vol': 0.0},
 'Crisis': {'trend': 0.0,
  'triple_coint': 1.0,
  'futures_trend': 0.0,
  'triple_vol': 0.0}}

In [10]:
results_df

,score,Sharpe,CAGR,MaxDD,weights
35,2.573213,2.573213,0.311654,-0.096126,"{'Stable Risk-On': {'trend': 1.0, 'triple_coin..."
43,2.568490,2.568490,0.319850,-0.096452,"{'Stable Risk-On': {'trend': 1.0, 'triple_coin..."
34,2.567914,2.567914,0.301259,-0.096381,"{'Stable Risk-On': {'trend': 1.0, 'triple_coin..."
36,2.564696,2.564696,0.322035,-0.095897,"{'Stable Risk-On': {'trend': 1.0, 'triple_coin..."
44,2.562836,2.562836,0.330299,-0.096223,"{'Stable Risk-On': {'trend': 1.0, 'triple_coin..."
42,2.560728,2.560728,0.309386,-0.096707,"{'Stable Risk-On': {'trend': 1.0, 'triple_coin..."
26,2.558962,2.558962,0.303392,-0.095800,"{'Stable Risk-On': {'trend': 1.0, 'triple_coin..."
25,2.555266,2.555266,0.293065,-0.096055,"{'Stable Risk-On': {'trend': 1.0, 'triple_coin..."
27,2.548650,2.548650,0.313704,-0.095571,"{'Stable Risk-On': {'trend': 1.0, 'triple_coin..."
50,2.547532,2.547532,0.327976,-0.096778,"{'Stable Risk-On': {'trend': 1.0, 'triple_coin..."
